<a href="https://colab.research.google.com/github/LUMII-AILab/NLP_Course/blob/main/notebooks/HFST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finite State Automata for Morphology

Akhil Kammalan Kandy
AK23204

Helsinki Finite State Transducers (HFST)

See also: https://github.com/hfst/compmorph-course/tree/v1.0

In [1]:
!pip install hfst==3.16
!pip install hfst_dev
!pip install graphviz

ERROR: Could not find a version that satisfies the requirement hfst_dev (from versions: none)
ERROR: No matching distribution found for hfst_dev


In [16]:
# Check if HFST is available
import sys
try:
    import hfst
except ImportError:
    print("HFST not found in environment")

In [28]:
import hfst
#import hfst_dev
import graphviz
from IPython.display import Image

In [36]:
# Load the Malayalam lexicon file
# The file should be in the same directory as this notebook
# If running in Google Colab, uncomment the wget command below:
!wget https://github.com/akhil-kk15/FundamentalsofNLP/blob/main/Malayalam.lexc

--2026-03-25 23:51:27--  https://github.com/akhil-kk15/FundamentalsofNLP/blob/main/Malayalam.lexc
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘Malayalam.lexc.1’

Malayalam.lexc.1        [ <=>                ] 240.37K  --.-KB/s    in 0.1s    

2026-03-25 23:51:28 (1.69 MB/s) - ‘Malayalam.lexc.1’ saved [246137]



## Malayalam Morphology FST

This notebook implements a Finite State Transducer for Malayalam morphology with:
- **3 noun inflection patterns**: Nominative (base), Genitive (^inte), Plural (^kal), Plural Genitive (^kalude)
- **2 adjective patterns**: Predicative (^aanu), Comparative (^kkaal)  
- **1 verb pattern**: Present tense (^unnu)
- **4 transformation rules**: Insertion, substitution (×2), removal

In [19]:
from hfst import compile_lexc_file
generator = hfst.compile_lexc_file('Malayalam.lexc')
print(generator.lookup('kutti+N+Sg+Gen'))
print(generator.lookup('maram+N+Sg+Gen'))
print(generator.lookup('kali+V+Pres'))
print (generator)


(('kutti^inte', 0.0),)
(('maram^inte', 0.0),)
(('kali^unnu', 0.0),)
0	1	n	n	0
0	2	k	k	0
0	3	p	p	0
0	4	v	v	0
0	5	c	c	0
0	6	m	m	0
1	7	a	a	0
2	8	a	a	0
2	9	u	u	0
3	10	a	a	0
3	11	u	u	0
4	12	a	a	0
4	13	e	e	0
5	14	h	h	0
6	15	a	a	0
7	16	l	l	0
7	17	d	d	0
8	18	l	l	0
9	19	t	t	0
10	20	a	a	0
11	21	s	s	0
12	22	a	a	0
12	23	l	l	0
13	24	e	e	0
14	25	e	e	0
15	26	r	r	0
16	27	l	l	0
17	28	a	a	0
18	29	i	i	0
19	30	t	t	0
20	18	d	d	0
21	31	t	t	0
22	32	y	y	0
23	33	i	i	0
24	34	d	d	0
25	23	r	r	0
26	35	a	a	0
27	36	a	a	0
28	37	k	k	0
29	38	+V	^	0
30	39	i	i	0
31	40	a	a	0
32	28	i	i	0
33	27	y	y	0
34	39	u	u	0
35	39	m	m	0
36	41	+Adj	^	0
37	29	k	k	0
38	42	+Pres	u	0
39	43	+N	@0@	0
39	44	+N	^	0
40	26	k	k	0
41	45	+Comp	k	0
41	46	+Pred	a	0
42	47	@0@	n	0
43	48	+Sg	@0@	0
44	49	+Pl	k	0
44	50	+Sg	i	0
45	51	@0@	k	0
46	47	@0@	a	0
47	52	@0@	n	0
48	53	+Nom	@0@	0
49	54	+Nom	a	0
49	55	+Gen	a	0
50	56	+Gen	n	0
51	57	@0@	a	0
52	53	@0@	u	0
53	0
54	53	@0@	l	0
55	58	@0@	l	0
56	59	@0@	t	0
57	54	@0@	a	0
58	60	@0@	u	0
59	53	@0@	e	0
60	59	@0@	d	0

## Transformation Rules and FST Composition

In [20]:
# Synthesis
from hfst import HfstTransducer

analyzer = HfstTransducer(generator) # Create a copy

analyzer.invert() # Invert the transducer to create an analyzer
analyzer.minimize() # Minimize the transducer for efficiency
print(analyzer.lookup('kutti^intr'))

()


### 1. Insertion Rule: InsertY

In [21]:
# Insertion rule: Insert 'y' between i and any vowel (with marker ^)
InsertY = hfst.regex('[..] -> y || i _ "^" [a|e|i|o|u]')
print("InsertY examples:")
print("  nalla^aanu →", InsertY.lookup("nalla^aanu"))

InsertY examples:
  nalla^aanu → (('nalla^aanu', 0.0),)


In [22]:
# Substitution rule 1: m -> th before genitive marker
MtoTH = hfst.regex('m -> t h || _ "^" i')
print("MtoTH examples:")
print("  maram^inte →", MtoTH.lookup("maram^inte"))
print("  pustakam^inte →", MtoTH.lookup("pustakam^inte"))

MtoTH examples:
  maram^inte → (('marath^inte', 0.0),)
  pustakam^inte → (('pustakath^inte', 0.0),)


### 2. Substitution Rules

### 3. Exception Handling

In [23]:
# Substitution rule 2: c -> č before plural marker
CReplacement = hfst.regex('c -> č || _ "^" M')
print("CReplacement test:", CReplacement.lookup("maram^Mkal"))

CReplacement test: (('maram^Mkal', 0.0),)


In [24]:
# Exception rule: Special morphophonemic rule for specific words
# Context: 'ai' → 'am' in dative case for certain roots (e.g., 'puik')
#linuguistic placeholder for the exception rule
Exceptions = hfst.regex('a i -> a m || p u i k "^" _')
print("Exception test:", Exceptions.lookup("puik^ai"))

Exception test: (('puik^am', 0.0),)


### 4. Removal/Cleanup Rules

In [25]:
# Removal rules: Clean up morpheme markers
MCleanup = hfst.regex('M -> 0')  # Remove plural marker
Cleanup = hfst.regex('"^" -> 0')  # Remove morpheme boundary
print("MCleanup test:", MCleanup.lookup("maram^Mkal"))

MCleanup test: (('maram^@_EPSILON_SYMBOL_@kal', 0.0),)


## FST Composition and Testing

In [33]:
# Compose the FST: generator pipeline
cascade = generator
cascade.compose(InsertY)
cascade.compose(MtoTH)
cascade.compose(CReplacement)
cascade.compose(Exceptions)
cascade.compose(MCleanup)
cascade.compose(Cleanup)

In [32]:
# Create analyzer (inverse of generator)
analyzer = hfst.HfstTransducer(cascade)
analyzer.invert()
analyzer.minimize()

## Test Results and Examples

In [ ]:
# Test generation with example words (NOT from lesson examples)
test_words = [
    # Noun inflections - 3 different stems
    "pustakam+N+Sg+Nom",  # book (nominative)
    "pustakam+N+Sg+Gen",  # book's (genitive)
    "pustakam+N+Pl+Nom",  # books (plural nominative)
    "veedu+N+Sg+Gen",     # house's (genitive)
    "veedu+N+Pl+Gen",     # houses' (plural genitive)
    
    # Adjective inflections - 3 different stems
    "valiya+Adj+Pred",    # is big (predicative)
    "valiya+Adj+Comp",    # more big (comparative)
    "cheriya+Adj+Pred",   # is small (predicative)
    "cheriya+Adj+Comp",   # more small (comparative)
    
    # Verb inflections - 3 different stems
    "paadi+V+Pres",       # singing -present
    "nadakk+V+Pres",      # walking -present
    "vaayikk+V+Pres",     # reading -present
]

print("Generated morphological forms:")
print("-" * 50)
for word in test_words:
    result = cascade.lookup(word)
    if result:
        output = result[0][0]
        print(f"{word:25} → {output}")
    else:
        print(f"{word:25} → (no output)")

Generated morphological forms:
--------------------------------------------------
pustakam+N+Sg+Nom         → pustakam@_EPSILON_SYMBOL_@@_EPSILON_SYMBOL_@@_EPSILON_SYMBOL_@
pustakam+N+Sg+Gen         → pustakath@_EPSILON_SYMBOL_@inte
pustakam+N+Pl+Nom         → pustakam@_EPSILON_SYMBOL_@kal
veedu+N+Sg+Gen            → veedu@_EPSILON_SYMBOL_@inte
veedu+N+Pl+Gen            → veedu@_EPSILON_SYMBOL_@kalude
valiya+Adj+Pred           → valiya@_EPSILON_SYMBOL_@aanu
valiya+Adj+Comp           → valiya@_EPSILON_SYMBOL_@kkaal
cheriya+Adj+Pred          → cheriya@_EPSILON_SYMBOL_@aanu
cheriya+Adj+Comp          → cheriya@_EPSILON_SYMBOL_@kkaal
paadi+V+Pres              → paadiy@_EPSILON_SYMBOL_@unnu
nadakk+V+Pres             → nadakk@_EPSILON_SYMBOL_@unnu
vaayikk+V+Pres            → vaayikk@_EPSILON_SYMBOL_@unnu


## Documentation: Malayalam Morphology FST

### Overview
This FST implements morphological generation and analysis for Malayalam, a South Indian language with rich morphological features.

### Lexicon Structure (Malayalam.lexc)
- **Noun stems**: kutti (child), maram (tree), pustakam (book), veedu (house)
- **Adjective stems**: nalla (good), valiya (big), cheriya (small)
- **Verb stems**: kali (play), paadi (sing), nadakk (walk), vaayikk (read)

### Inflection Patterns

#### Nouns (4 cases)
- **Nominative**: base form (kutti)
- **Genitive**: -inte suffix (kuttiyinte = child's)
- **Plural Nominative**: -kal suffix (kuttikal = children)
- **Plural Genitive**: -kalude suffix (kuttikalude = children's)

#### Adjectives (2 forms)
- **Predicative**: -aanu suffix (nallaanu = is good)
- **Comparative**: -kkaal suffix (nallakkaal = more good)

#### Verbs (1 form)
- **Present tense**: -unnu suffix (kaliyunnu = is playing)

### Transformation Rules

1. **Insertion Rule** (InsertY): Insert glide 'y' between 'i' and following vowel
   - Example: nalla^aanu → nallayaanu

2. **Substitution Rules** (MtoTH, CReplacement): 
   - m → th before genitive marker
   - c → č before plural marker

3. **Exception Rule**: Special morphophonemic rule for words like "puik"
   - ai → am in dative context

4. **Cleanup Rules**: Remove morpheme boundaries (^) and markers (M)
   - ^→0, M→0

### Composition Pipeline
The FST chain: Lexicon → InsertY → MtoTH → CReplacement → Exceptions → MCleanup → Cleanup

This produces correct surface forms by applying rules in sequence.